# Step 8: Model Training and Hyperparameter Tuning

## Purpose
This notebook improves the shortlisted churn models by tuning important hyperparameters in a controlled and interpretable way.

## What this step answers
- How do we get the best version of each shortlisted model?
- Which settings improve recall without destroying precision?
- Are we seeing signs of overfitting?
- Which tuned candidates deserve to move forward?

## Why this step matters
Baseline modeling tells us which model families are promising. Tuning helps us find stronger versions of those models while still protecting against overfitting and unfair comparisons.

## Key rule
We still do **not** touch the test set here. Hyperparameter decisions are made using training data plus validation-based comparison only.

In [9]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. Rebuild the feature set and the split

We rebuild the exact same cleaned feature set used in earlier modeling steps.

## Why?
Hyperparameter tuning must be based on the same feature logic and same split strategy as the baseline comparison. Otherwise, we would not know whether improvement came from better tuning or from a changed data setup.

In [10]:
PROJECT_DIR = Path.cwd()
CLEANED_FILE = PROJECT_DIR / "WA_Fn-UseC_-Telco-Customer-Churn-cleaned.csv"
RANDOM_STATE = 42

df = pd.read_csv(CLEANED_FILE)

df['service_count'] = (
    (df['PhoneService'] == 'Yes').astype(int)
    + (df['MultipleLines'] == 'Yes').astype(int)
    + (df['InternetService'] != 'No').astype(int)
    + (df['OnlineSecurity'] == 'Yes').astype(int)
    + (df['OnlineBackup'] == 'Yes').astype(int)
    + (df['DeviceProtection'] == 'Yes').astype(int)
    + (df['TechSupport'] == 'Yes').astype(int)
    + (df['StreamingTV'] == 'Yes').astype(int)
    + (df['StreamingMovies'] == 'Yes').astype(int)
)

df['avg_monthly_value_from_total'] = (df['TotalCharges'] / df['tenure'].replace(0, 1)).round(2)
df['is_new_customer'] = (df['tenure'] <= 12).astype(int)
df['has_long_term_contract'] = df['Contract'].isin(['One year', 'Two year']).astype(int)

X = df.drop(columns=['customerID', 'Churn']).copy()
y = df['Churn'].map({'No': 0, 'Yes': 1}).copy()

categorical_features = X.select_dtypes(include='object').columns.tolist()
numeric_features = X.select_dtypes(exclude='object').columns.tolist()

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_train_full,
)

X_train.shape, X_val.shape, X_test.shape

((4218, 23), (1407, 23), (1407, 23))

## 2. Define a shared preprocessing pipeline

We keep one preprocessing design for both tuned models.

## Why?
This keeps the comparison fair. If preprocessing changes from one candidate to another, then the results mix together two effects: preprocessing choices and model choices.

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 3. Define the shortlisted models and tuning grids

From Step 7, the two strongest candidates were:
- **Logistic Regression** for interpretation
- **Random Forest** for non-linear comparison

## Why tune only these two?
Because tuning should be focused. We do not want to waste effort optimizing models that already looked weaker in the baseline stage.

We also keep the grids modest in size.

## Why?
A huge search space may overfit the validation process and make learning harder. A smaller grid is often better for understanding which settings matter.

In [12]:
search_configs = {
    'Logistic Regression': {
        'pipeline': Pipeline([
            ('preprocessor', preprocessor),
            ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]),
        'param_grid': {
            'model__C': [0.01, 0.1, 1.0, 5.0, 10.0],
            'model__class_weight': [None, 'balanced'],
            'model__solver': ['lbfgs'],
        },
    },
    'Random Forest': {
        'pipeline': Pipeline([
            ('preprocessor', preprocessor),
            ('model', RandomForestClassifier(random_state=RANDOM_STATE)),
        ]),
        'param_grid': {
            'model__n_estimators': [100, 200],
            'model__max_depth': [6, 8, 12, None],
            'model__min_samples_leaf': [1, 3, 5],
            'model__class_weight': [None, 'balanced'],
        },
    },
}

## 4. Define evaluation helpers

We want to track both **training performance** and **validation performance**.

## Why?
If training metrics are much better than validation metrics, that is a warning sign for overfitting.

We also keep `recall`, `f1`, and `roc_auc` together.

## Why?
- `recall` matters because we want to catch churners
- `f1` helps balance recall with precision
- `roc_auc` tells us whether the model ranks risk well overall

In [13]:
def collect_metrics(model, X_data, y_data, split_name):
    y_pred = model.predict(X_data)

    if hasattr(model, 'predict_proba'):
        y_scores = model.predict_proba(X_data)[:, 1]
    else:
        y_scores = y_pred

    return {
        'split': split_name,
        'accuracy': round(accuracy_score(y_data, y_pred), 4),
        'precision': round(precision_score(y_data, y_pred, zero_division=0), 4),
        'recall': round(recall_score(y_data, y_pred, zero_division=0), 4),
        'f1': round(f1_score(y_data, y_pred, zero_division=0), 4),
        'roc_auc': round(roc_auc_score(y_data, y_scores), 4),
    }


def summarize_overfitting(train_metrics, val_metrics):
    return {
        'accuracy_gap': round(train_metrics['accuracy'] - val_metrics['accuracy'], 4),
        'precision_gap': round(train_metrics['precision'] - val_metrics['precision'], 4),
        'recall_gap': round(train_metrics['recall'] - val_metrics['recall'], 4),
        'f1_gap': round(train_metrics['f1'] - val_metrics['f1'], 4),
        'roc_auc_gap': round(train_metrics['roc_auc'] - val_metrics['roc_auc'], 4),
    }

## 5. Run the hyperparameter searches

We use `GridSearchCV` on the training set only.

## Why?
Cross-validation inside the training set helps us estimate which hyperparameters are more stable, without using the validation set to fit models directly.

After choosing the best settings from cross-validation, we then check the selected model on the validation set.

In [14]:
search_summaries = []
detailed_metrics = []
best_models = {}

for model_name, config in search_configs.items():
    search = GridSearchCV(
        estimator=config['pipeline'],
        param_grid=config['param_grid'],
        scoring='recall',
        cv=cv,
        n_jobs=-1,
        refit=True,
    )

    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    best_models[model_name] = best_model

    train_metrics = collect_metrics(best_model, X_train, y_train, 'train')
    val_metrics = collect_metrics(best_model, X_val, y_val, 'validation')
    gap_metrics = summarize_overfitting(train_metrics, val_metrics)

    search_summaries.append({
        'model': model_name,
        'best_cv_recall': round(search.best_score_, 4),
        'best_params': search.best_params_,
        **{f"train_{key}": value for key, value in train_metrics.items() if key != 'split'},
        **{f"val_{key}": value for key, value in val_metrics.items() if key != 'split'},
        **gap_metrics,
    })

    detailed_metrics.extend([
        {'model': model_name, **train_metrics},
        {'model': model_name, **val_metrics},
    ])

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [15]:
tuning_results_df = pd.DataFrame(search_summaries).sort_values(
    by=['val_recall', 'val_f1', 'val_roc_auc'],
    ascending=False,
)

detailed_metrics_df = pd.DataFrame(detailed_metrics)

tuning_results_df

,model,best_cv_recall,best_params,train_accuracy,train_precision,train_recall,train_f1,train_roc_auc,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,accuracy_gap,precision_gap,recall_gap,f1_gap,roc_auc_gap
1,Random Forest,0.8180,"{'model__class_weight': 'balanced', 'model__ma...",0.7677,0.5401,0.8475,0.6597,0.8800,0.7541,0.5252,0.7807,0.6280,0.8347,0.0136,0.0149,0.0668,0.0317,0.0453
0,Logistic Regression,0.8073,"{'model__C': 0.1, 'model__class_weight': 'bala...",0.7591,0.5306,0.8109,0.6415,0.8555,0.7534,0.5252,0.7513,0.6183,0.8343,0.0057,0.0054,0.0596,0.0232,0.0212


In [16]:
detailed_metrics_df

,model,split,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,train,0.7591,0.5306,0.8109,0.6415,0.8555
1,Logistic Regression,validation,0.7534,0.5252,0.7513,0.6183,0.8343
2,Random Forest,train,0.7677,0.5401,0.8475,0.6597,0.8800
3,Random Forest,validation,0.7541,0.5252,0.7807,0.6280,0.8347


## 6. How to interpret the tuning results

Read `tuning_results_df` with these questions in mind:

- Which model has the best **validation recall**?
- Does that model keep a reasonable **precision** and **f1**?
- Is the **training vs validation gap** small enough to trust?
- Are the best hyperparameters sensible, or do they suggest a very complex model?

## How to detect overfitting
Look at the gap columns:
- large positive gaps mean training performance is much better than validation performance
- smaller gaps usually mean the model generalizes better

## What usually happens here
- If **Logistic Regression** stays competitive after tuning, that is excellent for interpretation
- If **Random Forest** gains a lot but also shows large gaps, that may mean higher overfitting risk
- If recall improves mainly by using `class_weight='balanced'`, that tells us class imbalance handling matters

## 7. What to do after running this notebook

After you run the notebook, bring me these two outputs:

- `tuning_results_df`
- `detailed_metrics_df`

Then I will help you answer:
- which tuned model is the best candidate
- whether the model is overfitting
- whether the recall improvement is worth the trade-off
- which model should move forward to final evaluation and interpretation

# Step 8 notebook setup complete.
# Run the notebook from the top and review `tuning_results_df` and `detailed_metrics_df`.